### Logreader 

In [ ]:
%pip install -U langchain-ollama
%pip install chromadb


In [1]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    base_url="http://localhost:11434",
    model="gpt-oss:20b-cloud",
    temperature=0.5
)

Read log files 

In [2]:
# import os
# from langchain_core.tools import Tool

# @tool
# def summarise_logs() -> str:
#     "Read and return the summary of the first few lines from each log file in the logs directory."
#     log_directory = "./logs"
#     all_log = []
#     summary = ""
    
#     for filename in os.listdir(log_directory):
#         if filename.endswith(".log"):
#             with open(os.path.join(log_directory, filename), 'r') as file:
#                 log_content = file.read()
#                 prompt = f"Summarize the following log content:\n{log_content}\nSummary:"
#                 response = llm.invoke({"messages": [{"role": "user", "content": prompt}]})
#                 summary += f"Summary of {filename}:\n{response['output']}\n\n"
    
#     return summary

In [3]:
 ##I am using Visual Studio Code and I am trying to write the code for reading the log files. Here is my code:


import os
from langchain.tools import tool
#from langchain_community.tools import tool
from langchain_core.messages import HumanMessage
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama.embeddings import OllamaEmbeddings


@tool
def summarize_logs() -> str:
    "Read and return the summary of the first few lines from each log file in the logs directory."
    log_directory = "./logs"
    db_path ="./qa_db"
    all_logs = []
    summary = ""
    
    #logs reading
    for filename in os.listdir(log_directory):
        with open(os.path.join(log_directory, filename)) as f:
            all_logs.extend(f"{filename}:{line.strip()}" for line in f.readlines()[:200])
    summary =  "\n".join(all_logs) or "No log files found."

#Todo
#Check if DB exists. If so don't try to re-do the embeddings every time
    if not(os.path.exists(db_path) and os.path.isdir(db_path)):
        #Split the summary into chunks
        splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
        chunks = splitter.create_documents([summary])

        #embedding the chunks
        embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")
        db = Chroma.from_documents(chunks, embeddings, persist_directory="db_path")
        summary  += "\n\n [Embedding done and DB is created for future use.]"
    else:
        summary += "\n\n [Using existing DB for embeddings.]"
    return summary


In [4]:
import os
from langchain.agents import create_agent

@tool
def add_numbers(a: int, b: int) -> int:
    """Adds two numbers together."""
    return a + b   
@tool
def multiply_numbers(a: int, b: int) -> int:
    """Multiplies two numbers together."""
    return a * b
@tool
def subtract_numbers(a: int, b: int) -> int:
    """Subtracts the second number from the first."""
    return a - b
tools = [add_numbers, multiply_numbers, subtract_numbers, summarize_logs]
agent = create_agent(
    tools=tools,
    model=llm  
)



In [5]:
query = "what Are the errors in my logs? Please summarise them."
# result = agent.run(query)
# print("Final Result:", result)
response = agent.invoke({"messages": [HumanMessage(content=query)]})
print("Final Result:", response["messages"][-1].content)

Final Result: **Summary of error‑level events in the two log files**

| Log source | Timestamp | Component | Error message | Notes |
|------------|-----------|-----------|---------------|-------|
| **logs1.logs** | 10:00:03 | DBConnection | *Connection pool exhausted. Retrying…* | Retry succeeded; still a warning‑level error. |
|  | 10:00:12 | PaymentGateway | *Gateway timeout. 504 Gateway Time‑out* | Transaction failed; retried at 10:00:15. |
|  | 10:00:24 | Middleware | *Invalid JWT token signature* | Authentication failure; blocked IP already later. |
|  | 10:00:43 | Analytics | *Event dropped due to schema mismatch* | Data ingestion issue. |
|  | 10:01:30 | CDN | *Asset not found: /img/logo_v2.png* | 404 error served to front‑end. |
|  | 10:01:09 | Frontend | *JS Exception: undefined is not a function* | Client‑side script error. |
| **system.logs** | 10:00:21 | Kernel | *Out of memory: Kill process 2201 (python3)* | OOM killer triggered; service‑monitor crashed. |
|  | 10:00:23 | 

In [ ]:
# query = "what Are the errors in my logs? Please summarise them."
# result = agent.run(query)
# print("Final Result:", result)

AttributeError: 'CompiledStateGraph' object has no attribute 'run'